# Bigfoot Sightings

We are Timberline Media, a production company launching a new Bigfoot show called *Into the Treeline*. The network ordered a first season about the best Bigfoot sightings of all time, using reports going back to the 1860s. We have about 5,000 reports from across the country, but most of them don't work for our show.

Every report goes into one of three groups. A main segment costs $40,000 and gets a full reenactment, drone footage, and an on camera interview. A sub segment costs $8,000 and gets a two person crew recording audio and talking to locals. A scrap segment costs nothing because we don't pursue it.

Right now our two assistants read every report by hand, about 15 minutes each, and sort them into Class A, B, or C. Class A is supposed to mean our best stories. But it's their own judgment, they don't always agree, and it doesn't match what we actually need. During our pilot we booked a $40,000 shoot in Ohio and the witness admitted he only heard something.

In our dataset, the assistants' sorting is the `classification` column. Class A is meant for the best first-hand sightings, Class B for weaker encounters such as strange sounds or distant sightings, and Class C for the least credible stories.

So we made a checklist of what makes a good main segment. The witness has to have seen it themselves, not heard it or been told about it. Then we check five things: was anyone else there, was there good light, did it happen in 1990 or later so we can still interview them, is the story detailed enough (200+ words), and do we know where it happened. A visual sighting that passes 3 of the 5 is a main segment. A secondhand story, or anything that passes 1 or none, gets scrapped. Everything else is a sub segment.

We only need it once a year. We sort everything before Season 1, and if the show gets renewed we run each new year's reports through it at once before the next season. After each season we'll know which stories worked, and the system should learn from that and get better every year.


# Frame the problem and look at the big picture

**1. Define the objective in business terms.**

Timberline Media needs to sort about 5,000 Bigfoot reports into main segments ($40,000), sub segments ($8,000), and scrap segments ($0) for Season 1 of *Into the Treeline*. The goal is to stop wasting $40,000 shoots on weak stories, still find enough strong ones, and do it better than the assistants Class A, B, C sorting, which only matches what the show needs about half of the time. It has to be ready before production starts in January.

**2. How will your solution be used?**

The system runs once a year. Before Season 1 it sorts the whole archive. If the show is renewed, it sorts each new year's reports in one run before the next season. Each report comes out as `main_segment`, `sub_segment`, or `scrap_segment`.

**3. What are the current solutions/workarounds (if any)?**

Two assistants read every report by hand, about 15 minutes each, which is about 1,200 hours for the archive. They sort reports into Class A, B, or C based on their own judgment, and they don't always agree. Their Class A picks only line up with the show's checklist about half of the time. 

**4. How should you frame this problem (supervised/unsupervised, online/offline, …)?**

- Supervised learning: every report gets a segment label from the show's checklist 
- Multiclass classification: three classes, `main_segment`, `sub_segment`, and `scrap_segment`.
- Offline / batch learning: the model is trained once and used once a year on a whole batch of reports. 

**5. How should performance be measured? Is the performance measure aligned with the business objective?**

Our primary measure will be precision for `main_segment`: of the reports recommended for a $40,000 main shoot, how many actually meet the producers' checklist? A false main recommendation is especially costly, so this measure matches the client's main concern. We will also report `main_segment` recall, the number of main recommendations, a confusion matrix, and precision, recall, and F1 for all three segments. Finally, we will estimate the wasted budget on the test set (false main recommendations × $40,000 + false sub recommendations × $8,000), so the client can see the results in dollars. Producers will review recommendations before approving a shoot.


**6. What would be the minimum performance needed to reach the business objective?**

Our target is 75% `main_segment` precision on held-out reports, and the minimum we would accept is 70%. To see why, suppose the season needs 20 main segments. At 75% precision we would expect about 5 bad main shoots (about $200,000 wasted). At 70% we would expect about 6 (about $240,000). The assistants' rate of roughly 57% would mean about 9 bad shoots (about $360,000). We set the minimum at 70% because it is a clear improvement over the assistants and producers review every main recommendation before a shoot is booked, so some mistakes are caught before any money is spent. The system must also find enough main candidates to fill the season, so we will check recall and the number of reports recommended, not only precision.

The story estimates that the assistants' Class A picks match the checklist about half of the time (roughly 57%). With our data we can measure this directly: after creating the checklist labels, we will check how often the `classification` column agrees with them. This gives us a real baseline that the model must beat. We will also compare the model with other simple baselines: always predicting the most common segment (if about half of the reports are main candidates, this alone would reach about 50% precision), and a simple keyword rule (for example "saw" vs. "heard" in the report). The model is only worth using if it clearly beats these.

**7. What are comparable problems? Can you reuse experience or tools?**

This resembles editorial story screening and other tasks that route incoming reports for human review. We can reuse supervised text classification methods for the written reports and combine them with available structured information, such as dates and locations. Confusion matrices and per-class scores will help us see which kinds of routing mistakes the model makes.

**8. Is human expertise available?**

Yes. The producers defined the segment checklist and can resolve unclear cases. The assistants can explain the current manual process and help establish a comparison on the same reports. A producer will make the final decision before any costly shoot.

**9. How would you solve the problem manually?**

An assistant would read each report and decide whether it describes a first-hand visual sighting, a second-hand story, or another type of experience. For a first-hand visual report, the assistant would check for other witnesses, good lighting, a sighting date in or after 1990, at least 200 words of detail, and a known location. A visual report meeting at least three of these five checks becomes a `main_segment` candidate. A second-hand report, or a report meeting no more than one check, becomes `scrap_segment`. The remaining reports become `sub_segment`. Producers would review main candidates before scheduling a shoot.

The "good light" check can use more than the witness's own words. Besides the `time_and_conditions` text, the dataset includes weather for the day of the sighting, such as `moon_phase`, `cloud_cover`, `visibility`, and `precip_intensity`. A night sighting under a full moon with clear skies is more believable than one on a cloudy, moonless night.

**10. List the assumptions you (or others) have made so far. Verify assumptions if possible.**

- The producers' checklist represents what makes a report useful for the show. We will treat this as the client's decision rule, then ask producers to review unclear examples.

- The necessary checklist information is present in enough reports. We will inspect missing values and report text before creating labels; missing information will not automatically mean a check failed.

- A sighting from 1990 or later may have an interviewable witness, but the date alone cannot confirm this. Staff must verify contact and availability.

- The assistant accuracy estimate and the planned $40,000 and $8,000 costs are part of the fictional client scenario. We will distinguish them from values measured in the dataset.

- In the real dataset, the `classification` labels were assigned by BFRO investigators using written definitions (see the [BFRO classification guide](https://www.bfro.net/GDB/classify.asp)). For our client story, we treat them as the assistants' sorting. These labels describe the potential for misinterpreting a report, not whether a sighting was genuine or whether it fits the show.

- Historical BFRO reports may differ from new viewer submissions. Results on this dataset will be a retrospective test; the production team must continue reviewing new submissions. 

- Several checklist items cannot be read directly from a single column. Whether the witness saw the creature themselves, whether the light was good, and whether anyone else was there are written in free text (`observed`, `time_and_conditions`, `other_witnesses`). We still need to decide how these checks will be labeled (for example, by hand-labeling a sample of reports or by keyword rules that we check by hand). This is the hardest part of the project and the main reason a learned model is useful instead of a simple rule.

- Some checks (sighting in 1990 or later, 200+ words, known location) are computed directly from columns that could also be features. If we use them as features, the model can partly copy the rule. We will watch for this kind of leakage and check whether the model adds value beyond these simple checks.


# Get the data

1. List the data you need and how much you need
    * Bfro Sightings data 

2. Find and document where you can get that data
    * Kaggle [https://www.kaggle.com/datasets/thedevastator/unlocking-mysteries-of-bigfoot-through-sightings/data] 
    * Bfro Docs [https://www.bfro.net/gdb/classify.asp]
3. Get access authorizations 
    * Public dataset (free use)
4. Create a workspace (with enough storage space)
    * We will use our local project folder to run the notebooks and cache downloaded data for reuse.
5. Get the data
    * We provide functions to download and cache the dataset 
6. Convert the data to a format you can easily manipulate (without changing the data itself)
    * Changed to parquet files
7. Ensure sensitive information is deleted or protected (e.g. anonymized)
    * Data is anonymized 

# Functions

In [1]:
URL = "thedevastator/unlocking-mysteries-of-bigfoot-through-sightings"

# Extra columns from bfro_reports.csv that the geocoded file doesn't have
EXTRA_COLUMNS = [
    "YEAR", "MONTH", "NEAREST_TOWN", "NEAREST_ROAD", "ALSO_NOTICED",
    "OTHER_WITNESSES", "OTHER_STORIES", "TIME_AND_CONDITIONS", "ENVIRONMENT",
]

def list_dataset_files():
    """Prints the files in the Kaggle dataset so you know which names to use."""
    cache_dir = kagglehub.dataset_download(URL)
    files = sorted(os.listdir(cache_dir))
    for f in files:
        print(f)
    return files

def get_local_file(filename):
    """
    Checks if filename exists locally. If not, fetches it from
    the Kaggle cache and copies it to your project directory.
    """
    if not os.path.exists(filename):
        print(f"Fetching {filename} from Kaggle...")
        cache_dir = kagglehub.dataset_download(URL)
        cached_file_path = os.path.join(cache_dir, filename)
        shutil.copy(cached_file_path, filename)
    return filename

def read_geocoded_reports():
    """Loads the BFRO reports with geocoding and weather (5,021 reports)."""
    return pd.read_csv(get_local_file("bfro_reports_geocoded.csv"), parse_dates=["date"])

def read_raw_reports():
    """Loads the raw BFRO report form answers (witnesses, conditions, environment, etc.)."""
    return pd.read_csv(get_local_file("bfro_reports.csv"))

def read_locations():
    """Loads the BFRO report coordinates and timestamps."""
    return pd.read_csv(get_local_file("bfro_locations.csv"), parse_dates=["timestamp"])

def load_bigfoot_data():
    """
    Combines all three files into one dataset, one row per report:
    - starts from the geocoded reports (story, location, weather)
    - adds the extra form answers from the raw reports, matched on report number
    - fills missing coordinates from the locations file where possible
    """
    geo = read_geocoded_reports().drop(columns=["index"], errors="ignore")
    geo["number"] = geo["number"].astype(int)

    raw = read_raw_reports().dropna(subset=["REPORT_NUMBER"])   # drops ~464 blank rows
    raw["REPORT_NUMBER"] = raw["REPORT_NUMBER"].astype(int)
    raw = raw[["REPORT_NUMBER"] + EXTRA_COLUMNS].rename(columns=str.lower)

    df = (geo.merge(raw, left_on="number", right_on="report_number", how="left")
             .drop(columns=["report_number"]))

    locs = read_locations().set_index("number")
    df["latitude"] = df["latitude"].fillna(df["number"].map(locs["latitude"]))
    df["longitude"] = df["longitude"].fillna(df["number"].map(locs["longitude"]))

    return df

# Imports

In [4]:
%pip install kagglehub scikit-learn matplotlib numpy ipywidgets pandas pyarrow seaborn 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import shutil
import kagglehub


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data

In [ ]:
bigfoot_data = load_bigfoot_data()
bigfoot_data.shape

Fetching bfro_reports_geocoded.csv from Kaggle...


100%|██████████| 8.92M/8.92M [00:00<00:00, 14.1MB/s]

Extracting files...


Fetching bfro_reports.csv from Kaggle...
Fetching bfro_locations.csv from Kaggle...


(5021, 37)

In [6]:
bigfoot_data.head(10)

,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,wind_speed,year,month,nearest_town,nearest_road,also_noticed,other_witnesses,other_stories,time_and_conditions,environment
0,I was canoeing on the Sipsey river in Alabama....,NaN,Winston County,Alabama,Summer,NaN,NaN,NaN,NaT,30680,...,NaN,2010,NaN,Double Springs,Highway 33,3 hours before on river while I paused to fix ...,none,"No, but I've had a couple of weird things happ...","Dusk, was a clear day in about the high 80s te...",River running through a very remote forest are...
1,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaT,1261,...,NaN,Early 1990's,NaN,NaN,NaN,NaN,On a commercial fishing boat at anchor at the ...,NaN,"Early Fall, in the early 1990's.",NaN
2,"While attending U.R.I in the Fall of 1974,I wo...","Great swamp area, Narragansett Indians",Washington County,Rhode Island,Fall,Report 6496: Bicycling student has night encou...,41.45000,-71.50000,1974-09-20,6496,...,6.92,1974,September,Wakefield,Perry Ave.,none,none,"White gorilla was seen in a gravel pit, by a f...","10:00pm, very dark cool night.","Typical new England. Oak,Pine and Maple trees...."
3,"Hello, My name is Doug and though I am very re...",I would rather not have exact location (listin...,York County,Pennsylvania,Summer,NaN,NaN,NaN,NaT,8000,...,NaN,72 or 73,NaN,York; more specifically Manchester twp.,Raintree Road,My friend noted that he had spent the night be...,3. Don't remember the name of the third. He wa...,Just noting that my friend spent the night bef...,Incident occured somewhere between the hours o...,NaN
4,It was May 1984. Two friends and I were up in ...,"Logging roads north west of Yamhill, OR, about...",Yamhill County,Oregon,Spring,NaN,NaN,NaN,NaT,703,...,NaN,1984,May,Yamhill,NW Fairdale,Not sure. It took me a full 5 years to be able...,There were two other wittness' other than me. ...,NaN,I know that it was May of 1984 because it was ...,Gravel logging road. Very dense fir forests. B...
5,My two children and I were returning from Altu...,The creature crossed from one cattle pasture t...,Washita County,Oklahoma,Fall,Report 9765: Motorist and children have daylig...,35.30110,-99.17020,1973-09-28,9765,...,8.41,1973,September,Burns Flat,Highway 44,I remember thinking that the pavement would be...,My two children also saw the creature as I dro...,NaN,"The weather was clear, sunny and hot, around 3...","Barb wire fenced cattle pasture, mostly grassy..."
6,I was staying the night with a friends of mine...,"Vincent, Ohio (In theWayne National Forest)",Washington County,Ohio,Summer,Report 4983: Sighting and vocalizations at a h...,39.38745,-81.67339,1971-08-01,4983,...,NaN,1971,NaN,Vincent,NaN,NaN,Watching TV,NaN,NaN,My friend's parents Mobile home is out on a ol...
7,Well last year I was night fishing 9/2010 at A...,Both sightings were on the North east corner o...,Westchester County,New York,Fall,Report 31940: Fishermen and Law Enforcement Of...,41.29873,-73.73717,2010-09-01,31940,...,1.01,2010,September,Somers NY,Lake road,not really the Bigfoot didnt care or maybe did...,NaN,no just mine with my buddy and XXXXX's encount...,the Sun was just starting to come out,Its a NYC reservior..
8,I grew up in Northwestern Nevada along the Cal...,"The Western face of Peavine Mountain, Northwes...",Washoe County,Nevada,Fall,Report 5692: Father and son hear vocalizations...,39.59200,-119.97500,1970-09-01,5692,...,4.01,1970,September,Reno,US 395,Nothing other than that the source of the soun...,Just my father and I.,This is the only one I know of on Peavine Moun...,The incident occured about 3:30 PM. Light was...,The incident occured at about 6000 feet elevat...
9,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaT,438,...,NaN,2000,September,Allumuchy,not sure of the road,nothing,"my fri

# Train Test Split

### Why we choose our train test split percentage